In [26]:
import re
import numpy as np
import pandas as pd
import argparse
import sys
import glob
from pathlib import Path
from typing import Tuple, List, Dict
from datetime import datetime
import random
import json

In [27]:
def parse_c_header(file_path: str) -> Tuple[str, np.ndarray]:
    """
    Parse C header file and extract array name and data

    Args:
        file_path: Path to the C header file

    Returns:
        Tuple of (array_name, data_array)
    """
    with open(file_path, "r") as f:
        content = f.read()

    # Find array declaration
    array_pattern = r"unsigned\s+char\s+([a-zA-Z0-9_]+)\s*\[\]\s*=\s*\{"
    match = re.search(array_pattern, content)

    if not match:
        raise ValueError(f"Could not find array declaration in {file_path}")

    array_name = match.group(1)

    # Extract hex values
    hex_pattern = r"0x([0-9a-fA-F]{2})"
    hex_values = re.findall(hex_pattern, content)

    if not hex_values:
        raise ValueError(f"Could not find hex values in {file_path}")

    # Convert to numpy array
    data = np.array([int(h, 16) for h in hex_values], dtype=np.uint8)

    return array_name, data

In [28]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260402"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=3
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [29]:
CONFIG_PATH = Path(f"{TARGET_DIR}/{TARGET_DATE}/{TARGET_TYPE}/{TARGET_CONFIG}")

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

best = cfg["best_encoding"]
HUFFMAN_CODES = best["huffman_codes"]  # symbol(str) → bitstring
MERGED_VALUES = set(best["merged_values"])
REPRESENTATIVE_VALUE = best["representative_value"] # special rare-delta token
REPRESENTATIVE_CODE = HUFFMAN_CODES[str(REPRESENTATIVE_VALUE)]
BITSLENGTH = best["bitsLen_symNum"]
MIN_BITS_LENGTH = best["min_code_length"]
MAX_BITS_LENGTH = best["max_code_length"]

# Build reverse map  bitstring → int symbol  (for decoding)
REVERSE_CODES = {v: int(k) for k, v in HUFFMAN_CODES.items()}

print(f"[config]  total symbols in Huffman table : {len(HUFFMAN_CODES)}")
print(f"[config]  merged (rare) values           : {len(MERGED_VALUES)}")
print(f"[config]  representative code for {REPRESENTATIVE_VALUE}   : {REPRESENTATIVE_CODE}")
avg_bits = cfg["best_avg_bits"]
print(
    f"[config]  best avg bits/symbol           : {avg_bits:.4f}  "
    f"(vs 8 raw bits → {avg_bits/8*100:.1f}% of original)"
)

[config]  total symbols in Huffman table : 262
[config]  merged (rare) values           : 250
[config]  representative code for -194   : 000100
[config]  best avg bits/symbol           : 7.0287  (vs 8 raw bits → 87.9% of original)


In [30]:
print(REPRESENTATIVE_VALUE)
print(REPRESENTATIVE_CODE)

-194
000100


In [31]:
name1, data1 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID1}-quantized.h")
name2, data2 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID2}-quantized.h")

print(name1,"\n",name2)
print(f"Data 1 shape: {data1.shape}, Data 2 shape: {data2.shape}")

___models_1_LeNet_300_100_quantized_LeNet_300_100_1_quantized_tflite 
 ___models_1_LeNet_300_100_quantized_LeNet_300_100_3_quantized_tflite
Data 1 shape: (270720,), Data 2 shape: (270720,)


In [32]:
import os
base_name = os.path.basename(f"{TARGET_DIR}/{TARGET_NAME}_{MODEL_ID1}-quantized.h")
match = re.search(r'^([A-Za-z]+)-(\d+)-(\d+)_([0-9]+)-quantized\.h$', base_name)
if match:
    model = f"{match.group(1)}_{match.group(2)}_{match.group(3)}"
    number = match.group(4)
    print(model, number)

LeNet_300_100 1


In [33]:
import re

strings = [
    "___models_1_LeNet_300_100_quantized_LeNet_300_100_9_quantized_tflite",
    "___models_1_LeNet_300_100_quantized_LeNet_300_100_10_quantized_tflite"
]

for s in strings:
    match = re.search(r'_(\d+)_quantized_tflite$', s)
    if match:
        print(match.group(1))

9
10


In [34]:
import re

strings = [
    "___models_1_LeNet_300_100_quantized_LeNet_300_100_9_quantized_tflite",
    "___models_1_LeNet_300_100_quantized_LeNet_300_100_10_quantized_tflite"
]

for s in strings:
    match = re.search(r'quantized_(.+?)_\d+_quantized_tflite$', s)
    if match:
        print(match.group(1))

LeNet_300_100
LeNet_300_100


In [35]:
match = re.search(r'_(\d+)_quantized_tflite$', "__quantized_LeNet_300_100_1_quantized_tflite")
if match:
    print(match.group(1))

1


In [36]:
deltas = data2.astype(np.int16) - data1.astype(np.int16)
assert len(deltas) == len(data2.flat), (
    f"Length mismatch: delta={len(deltas)}, model2={len(data2.flat)}"
)

In [37]:
import math
import zlib

# ── 1. Build raw bitstring
bits = []
huffman_hits = 0
fallback_hits = 0

for d, w2 in zip(deltas, data2.flat):
    key = str(int(d))
    if key in HUFFMAN_CODES and key != str(REPRESENTATIVE_VALUE):
        bits.append(HUFFMAN_CODES[key])
        huffman_hits += 1
    else:
        bits.append(REPRESENTATIVE_CODE)
        bits.append(f"{int(w2) & 0xFF:08b}")
        fallback_hits += 1

bitstring = "".join(bits)

# ── 2. Determine frame count and target size
# Each frame = 8 bytes. Last 4 bytes of the whole package = CRC-32.
# Payload capacity = (n * 8) - 4 bytes = (n * 64) - 32 bits.

def compute_n(bitstring: str) -> int:
    """Minimum number of 8-byte frames to hold the bitstring + 4-byte CRC."""
    raw_bytes = math.ceil(len(bitstring) / 8)       # bytes needed for bits alone
    total_bytes = raw_bytes + 4                     # reserve 4 bytes for CRC
    return math.ceil(total_bytes / 8)               # round up to full frames

nb_frames = compute_n(bitstring)
target_bits = (nb_frames * 8 * 8) - 32              # total bits minus 32 CRC bits

# ── 3. Pad bitstring to exactly target_bits
pad = target_bits - len(bitstring)
bitstring += "0" * pad                              # zero-pad to fill payload

# ── 4. Pack payload bits into bytes
payload = bytes(
    int(bitstring[i:i+8], 2)
    for i in range(0, len(bitstring), 8)
)                                                   # exactly (n*8 - 4) bytes

# ── 5. Append 4-byte CRC-32 placeholder, compute, then overwrite
package = bytearray(payload) + bytearray(4)         # full n*8 bytes, CRC = 0x00000000
crc = zlib.crc32(bytes(package)) & 0xFFFFFFFF       # unsigned 32-bit, over entire package
package[-4:] = crc.to_bytes(4, byteorder="little")  # overwrite placeholder with real CRC

package = bytes(package)                            # final immutable bytes

# ── 6. Sanity checks
assert len(package) % 8 == 0,          "Package must be a multiple of 8 bytes"
assert len(package) == nb_frames * 8,          "Package size must equal n * 8 bytes"
assert len(package) >= len(payload) + 4, "CRC bytes must fit within package"

In [38]:
print("nb_frames:", nb_frames)
print("bits string len:", len(bitstring))
print("padding:", pad)
print("payload bytes:", len(payload))
print("total package bytes:", len(package))

nb_frames: 29542
bits string len: 1890656
padding: 5
payload bytes: 236332
total package bytes: 236336


In [39]:
with open(f"{TARGET_DIR}/package_payload_{MODEL_ID1}_{MODEL_ID2}.bin", "wb") as f:
    f.write(package)

In [40]:
import hashlib

with open(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID2}-quantized.h", "rb") as f:
    md5_hex = hashlib.md5(f.read()).hexdigest()
    
md5_hex

'7da28fd69f0409eeca258480f4035872'

In [41]:
# Store information for header package
header = cfg.get("header_package")

if header is not None:
    print("Exists:", header)
else:
    cfg["header_package"] = {
        "V_T": 0,
        "U_T": 0,
        "current_ver": MODEL_ID1,
        "update_ver": MODEL_ID2,
        "nb_symbols": len(HUFFMAN_CODES),
        "min_code_length": MIN_BITS_LENGTH,
        "max_code_length": MAX_BITS_LENGTH,
        "rwid": 8,
        "swid": 9,
        "representative_value": REPRESENTATIVE_VALUE,
        "target_size": len(data2),
        "payload_bits_len": len(bitstring),
        "target_checksum": md5_hex,
    }

    # Save back to file
    with open(CONFIG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)

In [42]:
print(huffman_hits)
print(fallback_hits)

270407
313


In [43]:
import zlib
from typing import NamedTuple


def decode_huffman_package(
    package: bytes,
    model1_flat,  # model1 weights, flat, same order as encoder
    reverse_codes,  # bitstring → int symbol
):
    """
    Decode a Huffman-compressed package back into model2 weights.

    Package layout (n * 8 bytes total):
        [ payload bits ... | pad bits ] [ CRC-32 : 4 bytes ]

    CRC is skipped per spec. pad_bits must be stored externally by the encoder.
    """

    # ── 1. Strip CRC (last 4 bytes), keep payload
    payload_bytes = package[:-4]

    # ── 2. Bytes → bitstring, strip padding
    bitstring = "".join(f"{byte:08b}" for byte in payload_bytes)

    # ── 3. Build prefix trie for O(1)-per-bit decoding
    # Walk the bitstring one bit at a time, accumulating until we match a code.
    weights = []
    huffman_hits = 0
    fallback_hits = 0

    pos = 0
    total = len(bitstring)
    m1_idx = 0  # index into model1_flat

    while pos < total:
        # ── 3a. Read bits until a known code is matched
        buf = ""
        symbol = None

        while pos < total:
            buf += bitstring[pos]
            pos += 1

            if buf in reverse_codes:
                symbol = reverse_codes[buf]
                break

            # Guard: no valid code can be longer than the longest code in table
            if len(buf) > max(len(k) for k in reverse_codes):
                raise ValueError(
                    f"No matching Huffman code found at bit position {pos}, "
                    f"buffer={buf!r}"
                )

        if symbol is None:
            break  # trailing pad bits exhausted — done

        # ── 3b. Dispatch on symbol
        if buf == REPRESENTATIVE_CODE:
            # Next 8 bits = raw unsigned uint8 weight (no model1 involved)
            if pos + 8 > total:
                raise ValueError(f"Fallback at bit {pos} but fewer than 8 bits remain.")
            raw_bits = bitstring[pos : pos + 8]
            pos += 8
            weight = int(raw_bits, 2)  # uint8, 0–255
            weights.append(weight)
            fallback_hits += 1
        else:
            # Normal delta: weight = model1[i] + decoded_delta (clamped to uint8)
            delta = symbol
            base = int(model1_flat[m1_idx])
            weight = (base + delta) & 0xFF  # keep as uint8
            weights.append(weight)
            huffman_hits += 1

        m1_idx += 1

        # Stop once we've recovered as many weights as model1 has
        if m1_idx >= len(model1_flat):
            break

    return weights, huffman_hits, fallback_hits

In [44]:
weights, huffman_hits, fallback_hits = decode_huffman_package(
    package=package,  # the n*8 bytes from the encoder
    model1_flat=list(data1.flat),
    reverse_codes=REVERSE_CODES,
)

# Sanity check
assert huffman_hits + fallback_hits == len(data1.flat)

reconstructed = np.array(weights, dtype=np.uint8).reshape(data2.shape)

In [45]:
print(huffman_hits)
print(fallback_hits)

270407
313


In [46]:
type(reconstructed)

numpy.ndarray

In [47]:
if np.array_equal(data2, reconstructed):
    print("success")
else:
    print("fail")

success


# Package Treatment
## Header

In [48]:
import struct
import zlib
from dataclasses import dataclass

MAGIC = b"Q-HUFDEL"  # exactly 8 bytes — frame 1


# Data class representing the decoded header
@dataclass
class HufDelHeader:
    # Frame 2
    version_type: int  # V-T  (1 byte)
    update_type: int  # U-T  (1 byte)
    version_dest: int  # current version of dest file  (2 bytes, LE)
    version_update: int  # update version of dest file   (2 bytes, LE)
    num_huffman_symbols: int  # number of Huffman symbols (2 byte LE)

    # Frame 3
    min_code_len: int  # min code length of symbol      (1 byte)
    max_code_len: int  # max code length of symbol      (1 byte)
    delta_bits: int  # delta value bit-width, e.g. 8 (1 byte)
    replace_bits: int  # model2 replace value bit-width (1 byte)
    # ↑ 4 bytes used — 4 bytes reserved padding to fill frame 3

    # Frame 4: checksum placeholder (0xFF * 8) — algorithm TBD

    # Frame 5
    model2_size_bytes: int  # size of model2 in bytes        (4 bytes, LE)
    bitstream_size_bytes: int  # total bitstream payload bytes  (4 bytes, LE)


# Encoder
HEADER_FRAME_COUNT = 5
HEADER_SIZE = HEADER_FRAME_COUNT * 8  # 40 bytes


def encode_header(h: HufDelHeader) -> bytes:
    """
    Encode the 5-frame (40-byte) package header.

    Frame layout:
        [0]  Magic          "Q-HUFDEL"                          8 bytes
        [1]  V-T | U-T | ver_dest (LE,2) | ver_upd (LE,2)
             | n_symbols                                        8 bytes
        [2]  min_code | max_code | delta_bits | replace_bits
             | _pad (4)                                         8 bytes
        [3]  Checksum placeholder 0xFF*8                        8 bytes
        [4]  model2_size (LE,4) | bitstream_size (LE,4)         8 bytes
    """

    # ── Frame 1: magic ────────────────────────────────────────────────────────
    frame1 = MAGIC  # 8 bytes

    # ── Frame 2: version / type flags ────────────────────────────────────────
    frame2 = struct.pack(
        "<BBHHH",  # little-endian
        h.version_type,  # 1 byte  — V-T
        h.update_type,  # 1 byte  — U-T
        h.version_dest,  # 2 bytes — current dest version
        h.version_update,  # 2 bytes — update dest version
        h.num_huffman_symbols,  # 2 byte  — symbol count
    )  # = 8 bytes

    # ── Frame 3: code / bit-width metadata ───────────────────────────────────
    frame3 = struct.pack(
        "<BBBB4x",  # 4 bytes data + 4 bytes padding
        h.min_code_len,  # 1 byte
        h.max_code_len,  # 1 byte
        h.delta_bits,  # 1 byte
        h.replace_bits,  # 1 byte
    )  # = 8 bytes

    # ── Frame 4: checksum placeholder (TBD) ──────────────────────────────────
    frame4 = b"\xff" * 8  # 8 bytes, algorithm not yet decided

    # ── Frame 5: sizes ────────────────────────────────────────────────────────
    frame5 = struct.pack(
        "<II",  # two unsigned 32-bit LE integers
        h.model2_size_bytes,  # 4 bytes — model2 file size
        h.bitstream_size_bytes,  # 4 bytes — bitstream payload size
    )  # = 8 bytes

    header = frame1 + frame2 + frame3 + frame4 + frame5
    assert len(header) == HEADER_SIZE, f"Header must be {HEADER_SIZE} bytes"
    return header


# Decoder
def decode_header(data: bytes) -> tuple[HufDelHeader, bytes]:
    """
    Parse the 40-byte header from the front of a package.

    Returns:
        (HufDelHeader, remainder)  — remainder is everything after the header.

    Raises:
        ValueError  on magic mismatch or truncated data.
    """
    if len(data) < HEADER_SIZE:
        raise ValueError(
            f"Package too short: expected at least {HEADER_SIZE} bytes, "
            f"got {len(data)}."
        )

    # ── Frame 1: magic check ──────────────────────────────────────────────────
    magic = data[0:8]
    if magic != MAGIC:
        raise ValueError(f"Magic mismatch: expected {MAGIC!r}, got {magic!r}")

    # ── Frame 2: version / type flags ────────────────────────────────────────
    (
        version_type,
        update_type,
        version_dest,
        version_update,
        num_huffman_symbols,
        _pad2,  # reserved — ignored
    ) = struct.unpack_from("<BBHHBB", data, offset=8)

    # ── Frame 3: code / bit-width metadata ───────────────────────────────────
    (
        min_code_len,
        max_code_len,
        delta_bits,
        replace_bits,
    ) = struct.unpack_from("<BBBB", data, offset=16)
    # bytes 20–23 are reserved padding — skipped

    # ── Frame 4: checksum placeholder — skip, algorithm TBD ──────────────────
    # data[24:32] — preserved as-is for future use

    # ── Frame 5: sizes ────────────────────────────────────────────────────────
    (
        model2_size_bytes,
        bitstream_size_bytes,
    ) = struct.unpack_from("<II", data, offset=32)

    header = HufDelHeader(
        version_type=version_type,
        update_type=update_type,
        version_dest=version_dest,
        version_update=version_update,
        num_huffman_symbols=num_huffman_symbols,
        min_code_len=min_code_len,
        max_code_len=max_code_len,
        delta_bits=delta_bits,
        replace_bits=replace_bits,
        model2_size_bytes=model2_size_bytes,
        bitstream_size_bytes=bitstream_size_bytes,
    )

    remainder = data[HEADER_SIZE:]
    return header, remainder

In [49]:
header = HufDelHeader(
    version_type=0x01,
    update_type=0x00,
    version_dest=0x0100,  # e.g. v1.0
    version_update=0x0101,  # e.g. v1.1
    num_huffman_symbols=len(HUFFMAN_CODES),
    min_code_len=min(len(v) for v in HUFFMAN_CODES.values()),
    max_code_len=max(len(v) for v in HUFFMAN_CODES.values()),
    delta_bits=8,
    replace_bits=8,
    model2_size_bytes=data2.nbytes,
    bitstream_size_bytes=len(payload),  # (n*8 - 4) bytes from encoder
)

full_package = encode_header(header) + package  # header + bitstream + CRC

In [50]:
header, bitstream_with_crc = decode_header(full_package)

# Sanity checks
assert header.bitstream_size_bytes == len(bitstream_with_crc) - 4
assert header.model2_size_bytes == data2.nbytes

## Symbols Specification

In [51]:
import struct
import math
from dataclasses import dataclass

# ─────────────────────────────────────────────────────────────────────────────
# Symbol table encoder
# ─────────────────────────────────────────────────────────────────────────────


def encode_symbol_table(
    bitslen_symnum: dict[str, int],  # e.g. {"3": 1, "4": 2, ...}
    huffman_codes: dict[str, str],  # e.g. {"0": "000", "-1": "0010", ...}
) -> bytes:
    """
    Encode two sections:
        Section A — bits-length histogram frames
        Section B — symbol value frames (int16 LE, sorted by code length then value)

    Every frame is exactly 8 bytes. Padding = 0x00.

    Section A layout (1 byte per count, packed 8 per frame):
        [ count_for_len3 | count_for_len4 | ... ] padded to next 8-byte boundary

    Section B layout (2 bytes per symbol, packed 4 per frame):
        [ sym0_lo | sym0_hi | sym1_lo | sym1_hi | ... ] padded to next 8-byte boundary
    """

    # ── Section A: bits-length histogram ─────────────────────────────────────
    # Emit counts in ascending key order, 1 byte each
    sorted_counts = [
        v for _, v in sorted(bitslen_symnum.items(), key=lambda x: int(x[0]))
    ]
    section_a_raw = bytes(sorted_counts)  # 1 byte per entry

    pad_a = (8 - len(section_a_raw) % 8) % 8
    section_a = section_a_raw + b"\x00" * pad_a

    # ── Section B: symbol values ──────────────────────────────────────────────
    # Sort: primary = code length, secondary = integer symbol value
    sorted_symbols = sorted(
        huffman_codes.keys(), key=lambda k: (len(huffman_codes[k]), int(k))
    )

    # Pack as int16 LE, 4 symbols per frame (4 × 2 = 8 bytes)
    symbol_ints = [int(k) for k in sorted_symbols]
    section_b_raw = struct.pack(f"<{len(symbol_ints)}h", *symbol_ints)

    pad_b = (8 - len(section_b_raw) % 8) % 8
    section_b = section_b_raw + b"\x00" * pad_b

    return section_a + section_b


def symbol_table_frame_count(
    bitslen_symnum: dict[str, int],
    huffman_codes: dict[str, str],
) -> tuple[int, int]:
    """
    Returns (section_a_frames, section_b_frames) — useful for the decoder
    to know how many frames to consume for each section.
    """
    n_counts = len(bitslen_symnum)
    n_symbols = len(huffman_codes)

    frames_a = math.ceil(n_counts / 8)  # 1 byte  per count, 8 per frame
    frames_b = math.ceil(n_symbols / 4)  # 2 bytes per symbol, 4 per frame

    return frames_a, frames_b


# ─────────────────────────────────────────────────────────────────────────────
# Symbol table decoder
# ─────────────────────────────────────────────────────────────────────────────


def decode_symbol_table(
    data: bytes,
    bitslen_symnum: dict[str, int],  # from header — tells us how many counts
    num_symbols: int,  # from header field num_huffman_symbols
) -> tuple[dict[str, int], bytes]:
    """
    Decode section A and section B from the front of `data`.

    Returns:
        (symbol_order, remainder)

        symbol_order: dict  int_symbol_value → code_length
                      (enough to reconstruct HUFFMAN_CODES if paired with
                       a canonical code rebuilder, or use directly for decode)

        remainder: bytes after the symbol table — this is the bitstream payload.
    """

    n_counts = len(bitslen_symnum)
    n_symbols = num_symbols

    frames_a, frames_b = (
        math.ceil(n_counts / 8),
        math.ceil(n_symbols / 4),
    )
    size_a = frames_a * 8
    size_b = frames_b * 8

    if len(data) < size_a + size_b:
        raise ValueError(
            f"Symbol table too short: need {size_a + size_b} bytes, "
            f"got {len(data)}."
        )

    # ── Section A: read counts ────────────────────────────────────────────────
    section_a_bytes = data[:size_a]
    sorted_lengths = sorted(bitslen_symnum.keys(), key=int)  # ["3","4","5",...]
    counts = {
        int(bit_len): section_a_bytes[i] for i, bit_len in enumerate(sorted_lengths)
    }
    # counts = {3: 1, 4: 2, 5: 14, ...}

    # ── Section B: read symbol values ─────────────────────────────────────────
    section_b_bytes = data[size_a : size_a + size_b]
    symbol_ints = list(struct.unpack_from(f"<{n_symbols}h", section_b_bytes))
    # symbol_ints = [0, -1, 1, -8, -7, ...] in sorted order

    # Reconstruct symbol → code_length mapping using the histogram
    symbol_code_len: dict[int, int] = {}
    idx = 0
    for bit_len in sorted(counts.keys()):
        count = counts[bit_len]
        for sym in symbol_ints[idx : idx + count]:
            symbol_code_len[sym] = bit_len
        idx += count

    remainder = data[size_a + size_b :]
    return symbol_code_len, remainder


# ─────────────────────────────────────────────────────────────────────────────
# Full package assembler
# ─────────────────────────────────────────────────────────────────────────────


def assemble_package(
    header: HufDelHeader,
    bitslen_symnum: dict[str, int],
    huffman_codes: dict[str, str],
    payload: bytes,  # bitstream payload (n*8 - 4 bytes)
    crc: bytes,  # 4-byte CRC-32 (from encode step)
) -> bytes:
    """
    Final layout:
        [ 5 header frames (40 bytes)        ]
        [ section A: histogram frames       ]
        [ section B: symbol value frames    ]
        [ bitstream payload                 ]
        [ CRC-32 : 4 bytes                  ]
    """
    return (
        encode_header(header)
        + encode_symbol_table(bitslen_symnum, huffman_codes)
        + payload
        + crc
    )


# ─────────────────────────────────────────────────────────────────────────────
# Full package disassembler
# ─────────────────────────────────────────────────────────────────────────────


def disassemble_package(data: bytes) -> tuple[HufDelHeader, dict[int, int], bytes]:
    """
    Parse the entire package.

    Returns:
        (header, symbol_code_len, bitstream_with_crc)

        symbol_code_len : dict  int_symbol → code_length
        bitstream_with_crc : raw bytes of payload + 4-byte CRC at tail
    """

    # ── 1. Header ─────────────────────────────────────────────────────────────
    header, after_header = decode_header(data)

    # ── 2. Symbol table ───────────────────────────────────────────────────────
    # Reconstruct bitslen_symnum from header.num_huffman_symbols is not enough;
    # the histogram is embedded in section A — pass a sentinel to get frame sizes.
    # We need to read section A first to know counts, then validate against header.

    frames_a = math.ceil(header.num_huffman_symbols / 8)  # rough upper bound
    # Actually derive precisely: num_huffman_symbols tells us total symbols,
    # but we need the histogram key-count. We read section A raw, then decode.

    # The number of distinct code lengths = len(bitslen_symnum) — this is NOT
    # stored explicitly, so we derive it from min/max code length in the header.
    n_lengths = header.max_code_len - header.min_code_len + 1

    frames_a = math.ceil(n_lengths / 8)
    size_a = frames_a * 8
    size_b = math.ceil(header.num_huffman_symbols / 4) * 8

    # Rebuild bitslen_symnum from section A bytes + header min/max
    section_a_bytes = after_header[:size_a]
    bitslen_symnum = {
        str(header.min_code_len + i): section_a_bytes[i] for i in range(n_lengths)
    }

    symbol_code_len, bitstream_with_crc = decode_symbol_table(
        after_header,
        bitslen_symnum,
        header.num_huffman_symbols,
    )

    return header, symbol_code_len, bitstream_with_crc

In [52]:
frames_a, frames_b = symbol_table_frame_count(BITSLENGTH, HUFFMAN_CODES)

full = assemble_package(
    header=header,
    bitslen_symnum=BITSLENGTH,
    huffman_codes=HUFFMAN_CODES,
    payload=payload,  # (n*8 - 4) bytes from encoder
    crc=package[-4:],  # 4-byte CRC from encoder
)

In [53]:
import struct
import math

# ─────────────────────────────────────────────────────────────────────────────
# Encoder
# ─────────────────────────────────────────────────────────────────────────────

def encode_symbol_table(
    bitslen_symnum: dict[str, int],   # e.g. {"3": 1, "4": 2, "5": 14, ...}
    huffman_codes:  dict[str, str],   # e.g. {"0": "000", "-1": "0010", ...}
) -> bytes:
    """
    Encode histogram counts and symbol values into a single packed byte stream,
    framed at 8-byte boundaries, zero-padded at the end.

    Stream layout (before framing):
        [ count_0 (1B) | count_1 (1B) | ... | count_N (1B)
        | sym_0 (2B LE) | sym_1 (2B LE) | ... | sym_M (2B LE) ]
        + 0x00 padding to next 8-byte boundary

    counts are emitted in ascending code-length order.
    symbols are emitted sorted by (code_length ASC, symbol_value ASC).
    """

    # ── Counts: 1 byte each, ascending code-length order ─────────────────────
    sorted_counts = [
        v for _, v in sorted(bitslen_symnum.items(), key=lambda x: int(x[0]))
    ]
    count_bytes = bytes(sorted_counts)                      # 1 byte per count

    # ── Symbols: int16 LE, sorted by code length then symbol value ────────────
    sorted_symbols = sorted(
        huffman_codes.keys(),
        key=lambda k: (len(huffman_codes[k]), int(k))
    )
    symbol_ints  = [int(k) for k in sorted_symbols]
    symbol_bytes = struct.pack(f"<{len(symbol_ints)}h", *symbol_ints)

    # ── Concatenate then pad to 8-byte boundary ───────────────────────────────
    raw     = count_bytes + symbol_bytes
    pad_len = (8 - len(raw) % 8) % 8
    return raw + b"\x00" * pad_len


def symbol_table_byte_size(
    bitslen_symnum: dict[str, int],
    num_symbols:    int,
) -> int:
    """Total byte size of the encoded symbol table section (always a multiple of 8)."""
    n_counts  = len(bitslen_symnum)     # 1 byte each
    raw       = n_counts + num_symbols * 2
    return math.ceil(raw / 8) * 8


# ─────────────────────────────────────────────────────────────────────────────
# Decoder
# ─────────────────────────────────────────────────────────────────────────────

def decode_symbol_table(
    data:           bytes,
    min_code_len:   int,              # from header
    max_code_len:   int,              # from header
    num_symbols:    int,              # from header: num_huffman_symbols
) -> tuple[dict[int, int], bytes]:
    """
    Decode the combined symbol table section.

    Returns:
        (symbol_code_len, remainder)

        symbol_code_len : dict  int_symbol_value → code_length
        remainder       : bytes immediately after the symbol table (bitstream)
    """

    n_lengths = max_code_len - min_code_len + 1   # number of histogram entries
    n_counts  = n_lengths                          # 1 byte per length bucket
    n_symbols = num_symbols                        # 2 bytes per symbol

    total_raw  = n_counts + n_symbols * 2
    total_size = math.ceil(total_raw / 8) * 8      # padded to 8-byte boundary

    if len(data) < total_size:
        raise ValueError(
            f"Symbol table too short: need {total_size} bytes, got {len(data)}."
        )

    raw = data[:total_raw]                         # exclude padding bytes

    # ── Read counts ───────────────────────────────────────────────────────────
    count_bytes = raw[:n_counts]
    counts: dict[int, int] = {
        min_code_len + i: count_bytes[i]
        for i in range(n_lengths)
    }
    # e.g. {3: 1, 4: 2, 5: 14, ...}

    # ── Read symbol int16 values ──────────────────────────────────────────────
    symbol_bytes = raw[n_counts : n_counts + n_symbols * 2]
    symbol_ints  = list(struct.unpack(f"<{n_symbols}h", symbol_bytes))
    # e.g. [0, -1, 1, -8, -7, ..., 1000]

    # ── Map each symbol to its code length using the histogram ────────────────
    symbol_code_len: dict[int, int] = {}
    idx = 0
    for bit_len in sorted(counts.keys()):
        count = counts[bit_len]
        for sym in symbol_ints[idx : idx + count]:
            symbol_code_len[sym] = bit_len
        idx += count

    if idx != n_symbols:
        raise ValueError(
            f"Histogram total {idx} does not match num_symbols {n_symbols}."
        )

    remainder = data[total_size:]
    return symbol_code_len, remainder